In [27]:
import kagglehub
import pandas as pd
import numpy as np
import random
import math

# ==========================================
# 1. The Data and Preprocessing
# ==========================================
print("Downloading dataset from Kaggle...")
path = kagglehub.dataset_download("ifuurh/public-transportation-passenger-counts")
print("Path to dataset files:", path)

df = pd.read_parquet(path + '/public_transport.parquet')

# Convert columns to correct types
df['departure']      = pd.to_numeric(df['departure'],      errors='coerce')
df['arrival']        = pd.to_numeric(df['arrival'],        errors='coerce')
df['passengers']     = pd.to_numeric(df['passengers'],     errors='coerce')
df['pattern_index']  = pd.to_numeric(df['pattern_index'],  errors='coerce')
df['vehicle_seats']  = pd.to_numeric(df['vehicle_seats'],  errors='coerce')
df['line_id']           = df['line_id'].astype(str)
df['trip_direction']    = df['trip_direction'].astype(str)
df['stop_id_departure'] = df['stop_id_departure'].astype(str)
df['stop_id_end']       = df['stop_id_end'].astype(str)

# Filtering constants
TARGET_LINE        = '74'
TARGET_DIRECTION   = '1'
TARGET_ORIGIN      = '536'
TARGET_DESTINATION = '565'
START_TIME_SEC     = 7 * 3600   # 07:00 AM
END_TIME_SEC       = 8 * 3600   # 08:00 AM

print("\nFiltering to a single route branch in the morning rush hour...")
df_filtered = df[
    (df['line_id']           == TARGET_LINE)        &
    (df['trip_direction']    == TARGET_DIRECTION)   &
    (df['stop_id_departure'] == TARGET_ORIGIN)      &
    (df['stop_id_end']       == TARGET_DESTINATION) &
    (df['departure']         >= START_TIME_SEC)     &
    (df['departure']         <= END_TIME_SEC)
].copy()

df_filtered.dropna(subset=['pattern_index','departure','arrival','passengers','stop_id','trip_departure'], inplace=True)

# ==========================================
# The dataset has one row per PASSENGER TICKET, not per stop.
# Aggregate: collapse to ONE row per (trip, pattern_index=sequential stop).
# pattern_index is the true sequential stop counter within a single trip (1, 2, 3 ...)
# ==========================================
df_agg = (
    df_filtered
    .groupby(['trip_departure', 'pattern_index', 'stop_id'], as_index=False)
    .agg(
        arrival      =('arrival',      'first'),
        departure    =('departure',    'first'),
        passengers   =('passengers',   'max'),
        vehicle_seats=('vehicle_seats','first')
    )
)

df_agg.sort_values(['trip_departure', 'pattern_index'], inplace=True)

grouped_trips = df_agg.groupby('trip_departure')
print(f"\nTotal distinct bus runs found in this hour: {len(grouped_trips)}")

# ==========================================
# Show EVERY bus run, stop-by-stop
# ==========================================
print("\n--- Geographic Trace: Every Bus Moving From Stop To Stop ---")
for trip_id, trip_run in grouped_trips:
    dep_sec  = int(trip_id)
    dep_time = f"{dep_sec // 3600:02d}:{(dep_sec % 3600) // 60:02d}"
    print(f"\n{'='*65}")
    print(f"  Bus | Origin departure: {dep_time}  ({dep_sec}s from midnight)")
    print(f"{'='*65}")
    print(trip_run[['pattern_index','stop_id','arrival','departure','passengers','vehicle_seats']].to_string(index=False))

# ==========================================
# Extract canonical stop sequence for the simulator
# (use the trip with the most complete stop coverage)
# ==========================================
canonical_trip_id = df_agg.groupby('trip_departure')['pattern_index'].count().idxmax()
canonical_run     = df_agg[df_agg['trip_departure'] == canonical_trip_id].sort_values('pattern_index')
route_stops       = canonical_run['stop_id'].tolist()
print(f"\nCanonical route: {len(route_stops)} stops, from stop_id {route_stops[0]} → {route_stops[-1]}")

# ==========================================
# Precalculate per-stop simulation parameters
# ==========================================
print("\nCalculating inter-stop travel times and passenger demand rates...")

MAX_VEHICLE_SEATS = int(df_agg['vehicle_seats'].max()) if not pd.isna(df_agg['vehicle_seats'].max()) else 99
grouped_stops = df_agg.groupby('stop_id')
travel_times  = {}
arrival_rates = {}

for i, stop in enumerate(route_stops):
    # Inter-stop travel time from real historical averages
    if i < len(route_stops) - 1:
        next_stop = route_stops[i + 1]
        this_dep = df_agg[df_agg['stop_id'] == stop]['departure'].mean()
        next_arr = df_agg[df_agg['stop_id'] == next_stop]['arrival'].mean()
        tt = next_arr - this_dep
        travel_times[stop] = int(tt) if tt > 0 else 120
    else:
        travel_times[stop] = 0

    # Passenger demand rate: people arriving per second at this stop
    if stop in grouped_stops.groups:
        avg_pax = grouped_stops.get_group(stop)['passengers'].mean()
        arrival_rates[stop] = max(0.01, (avg_pax * 0.1) / 60)
    else:
        arrival_rates[stop] = 0.05


Path to dataset files: C:\Users\rares\.cache\kagglehub\datasets\ifuurh\public-transportation-passenger-counts\versions\1

Filtering to a single route branch in the morning rush hour...

Total distinct bus runs found in this hour: 6

--- Geographic Trace: Every Bus Moving From Stop To Stop ---

  Bus | Origin departure: 05:46  (20760s from midnight)
 pattern_index  stop_id  arrival  departure  passengers  vehicle_seats
            39      557    25200      25200          15           99.0
            40      559    25320      25320          15           99.0
            41      560    25380      25380          15           99.0
            42      561    25500      25740           9           99.0
            43      558    25860      25860           6           99.0
            44      562    25920      25920           6           99.0
            45      563    25980      25980           5           99.0
            46      564    26040      26040           5           99.0
          

In [32]:
import pandas as pd
import kagglehub

print("Downloading and loading dataset...")
path = kagglehub.dataset_download("ifuurh/public-transportation-passenger-counts")
df = pd.read_parquet(path + '/public_transport.parquet')

# 1. Convert columns to numeric so math works without errors
df['trip_departure'] = pd.to_numeric(df['trip_departure'], errors='coerce')
df['passengers'] = pd.to_numeric(df['passengers'], errors='coerce')
df['pattern_index'] = pd.to_numeric(df['pattern_index'], errors='coerce')

# 2. Filter for Line 74, Direction 1 FIRST (Ignoring time for a moment)
df_filtered = df[(df['line_id'] == '74') & (df['trip_direction'] == '1')].copy()
df_filtered.dropna(subset=['pattern_index', 'stop_id', 'passengers', 'trip_departure'], inplace=True)

# 3. FIX THE CAPACITY: Force all buses to be 99 seats so your assignment math is perfectly uniform
df_filtered['vehicle_seats'] = 99

# 4. FIX THE BUS LENGTHS: Find the most common "perfect" route length
trip_stop_counts = df_filtered.groupby('trip_departure')['stop_id'].count()
perfect_stop_count = trip_stop_counts.mode().iloc # Finds the standard, unbroken route length

# Keep ONLY the buses that perfectly completed this exact number of stops
perfect_buses = trip_stop_counts[trip_stop_counts == perfect_stop_count].index
df_perfect = df_filtered[df_filtered['trip_departure'].isin(perfect_buses)].copy()

# 5. NOW apply the time window (Buses that started between 6:00 AM and 9:00 AM)
# We start at 6:00 AM because a bus that launched at 6:30 AM is still driving during your 7-8 AM window!
df_final = df_perfect[
    (df_perfect['trip_departure'] >= 6 * 3600) & 
    (df_perfect['trip_departure'] <= 9 * 3600)
].copy()

# Sort the data chronologically (by bus launch time) and geographically (by stop sequence)
df_final = df_final.sort_values(by=['trip_departure', 'pattern_index'])

# ==========================================================
# PRINT THE RESULTS
# ==========================================================
grouped_perfect_buses = df_final.groupby('trip_departure')

print(f"\nSUCCESS! Found {len(grouped_perfect_buses)} perfectly identical buses running one after another.")

for bus_start_time, bus_data in grouped_perfect_buses:
    
    # Convert the starting seconds back into a readable HH:MM format
    start_hour = int(bus_start_time // 3600)
    start_minute = int((bus_start_time % 3600) // 60)
    
    print(f"\n=======================================================")
    print(f" BUS LAUNCHED FROM DEPOT AT {start_hour:02d}:{start_minute:02d} | Total Stops: {len(bus_data)} | Capacity: 99")
    print(f"=======================================================")
    
    # Select only the columns you actually want to see
    clean_view = bus_data[['pattern_index', 'stop_id', 'passengers']].copy()
    
    # Rename them so they make complete sense
    clean_view.rename(columns={
        'pattern_index': 'Stop Sequence',
        'stop_id': 'Bus Stop ID',
        'passengers': 'People Inside Bus'
    }, inplace=True)
    
    print(clean_view.to_string(index=False))


SUCCESS! Found 0 perfectly identical buses running one after another.


In [33]:
import pandas as pd
import kagglehub
import warnings
warnings.filterwarnings('ignore') # Keeps your console output clean

print("Downloading and loading dataset...")
path = kagglehub.dataset_download("ifuurh/public-transportation-passenger-counts")
df = pd.read_parquet(path + '/public_transport.parquet')

# 1. Convert columns to numeric
cols_to_numeric = ['departure', 'passengers', 'pattern_index', 'trip_departure']
for col in cols_to_numeric:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 2. Filter for Line 74, Direction 1
df_filtered = df[(df['line_id'] == '74') & (df['trip_direction'] == '1')].copy()
df_filtered.dropna(subset=['pattern_index', 'stop_id', 'passengers', 'trip_departure'], inplace=True)

# 3. FIX THE "MULTIPLE DAYS" ISSUE
# We average the passengers across the multiple days so we have one clean run per scheduled time
df_agg = df_filtered.groupby(['trip_departure', 'pattern_index', 'stop_id'], as_index=False).agg({
    'passengers': 'mean' 
})

# Force every single bus to have exactly 99 seats so your assignment math is perfectly uniform
df_agg['Capacity'] = 99

# 4. Filter for the morning rush hour (6:00 AM to 9:00 AM)
df_morning = df_agg[
    (df_agg['trip_departure'] >= 6 * 3600) & 
    (df_agg['trip_departure'] <= 9 * 3600)
].copy()

# 5. THE MAGIC FIX: Find the "Golden Route"
# Get the exact sequence of stop IDs for every bus
sequences = df_morning.groupby('trip_departure')['stop_id'].apply(tuple)

# Find the most common, perfect exact sequence of stops
golden_sequence = sequences.mode().iloc

# Keep ONLY the buses that follow this exact sequence perfectly
perfect_buses = sequences[sequences == golden_sequence].index
df_final = df_morning[df_morning['trip_departure'].isin(perfect_buses)].copy()

# Sort geographically and chronologically
df_final = df_final.sort_values(by=['trip_departure', 'pattern_index'])

# ==========================================================
# PRINT THE RESULTS
# ==========================================================
grouped_perfect_buses = df_final.groupby('trip_departure')

print(f"\nSUCCESS! Found {len(grouped_perfect_buses)} perfectly identical buses running one after another.")

for bus_start_time, bus_data in grouped_perfect_buses:
    start_hour = int(bus_start_time // 3600)
    start_minute = int((bus_start_time % 3600) // 60)
    
    print(f"\n=======================================================")
    print(f" BUS LAUNCHED FROM DEPOT AT {start_hour:02d}:{start_minute:02d} | Total Stops: {len(bus_data)} | Capacity: 99")
    print(f"=======================================================")
    
    clean_view = bus_data[['pattern_index', 'stop_id', 'passengers']].copy()
    clean_view['passengers'] = clean_view['passengers'].astype(int) # Round to whole people
    
    clean_view.rename(columns={
        'pattern_index': 'Stop Sequence',
        'stop_id': 'Bus Stop ID',
        'passengers': 'Avg People Inside'
    }, inplace=True)
    
    print(clean_view.to_string(index=False))


SUCCESS! Found 0 perfectly identical buses running one after another.


In [17]:

# ==========================================
# 2. The Simulation & Constants
# ==========================================

ALIGHTING_RATE = 0.20
FLEET_SIZE = 10 # Number of buses dispatched in this hour

def evaluate_timetable(schedule):
    """
    Fitness Function (Stop-by-Stop Geographic Simulation).
    A schedule is a list of exact departure times (in seconds since midnight) from the first stop.
    Returns the total cumulative sum of all stranded passengers across all stops.
    """
    # Sort the schedule chronologically
    schedule = sorted(schedule)
    total_stranded = 0
    
    # Track the last time a bus visited each stop to accumulate waiting passengers over time
    last_visit_time = {stop: START_TIME_SEC for stop in route_stops}
    
    for bus_start_time in schedule:
        current_time = bus_start_time
        current_passengers = 0
        
        for i, stop in enumerate(route_stops):
            # Advance time if moving to the next stop
            if i > 0:
                prev_stop = route_stops[i-1]
                current_time += travel_times.get(prev_stop, 120)
                
            # Apply ALIGHTING_RATE to free up seats
            alighting_count = int(current_passengers * ALIGHTING_RATE)
            current_passengers -= alighting_count
            
            # Calculate available capacity
            available_capacity = MAX_VEHICLE_SEATS - current_passengers
            
            # Calculate passengers waiting at the stop
            # Time elapsed since the last bus cleared this stop
            time_elapsed = max(0, current_time - last_visit_time[stop])
            waiting_passengers = int(time_elapsed * arrival_rates.get(stop, 0.05))
            
            # Board waiting passengers until capacity limit is reached
            boarded = min(waiting_passengers, available_capacity)
            current_passengers += boarded
            
            # Count stranded passengers (those who couldn't fit)
            stranded = waiting_passengers - boarded
            total_stranded += stranded
            
            # Update the last visit time for this stop to the current bus's arrival time
            # Note: Any stranded passengers theoretically carry over, but by resetting the timer, 
            # we count total stranded occurrences. Alternatively, we could carry them in a backlog.
            last_visit_time[stop] = current_time
            
    return total_stranded

# ==========================================
# 3. The Algorithms
# ==========================================

# --- Phase 1: Genetic Algorithm (GA) ---

def create_chromosome():
    """Generates an integer array representing exact departure times (seconds since midnight)."""
    return [random.randint(START_TIME_SEC, END_TIME_SEC) for _ in range(FLEET_SIZE)]

def crossover(parent1, parent2):
    """Single-point crossover."""
    pt = random.randint(1, FLEET_SIZE - 2)
    child1 = parent1[:pt] + parent2[pt:]
    child2 = parent2[:pt] + parent1[pt:]
    return child1, child2

def mutate(individual, mutation_rate=0.15):
    """Randomly assigns a new departure time to a gene."""
    for i in range(len(individual)):
        if random.random() < mutation_rate:
            individual[i] = random.randint(START_TIME_SEC, END_TIME_SEC)
    return individual

def run_genetic_algorithm(pop_size=50, generations=50):
    """Executes the GA to find a macro-schedule with low stranded passengers."""
    population = [create_chromosome() for _ in range(pop_size)]
    
    best_overall = None
    best_fitness = float('inf')
    
    for gen in range(generations):
        # Evaluate fitness (minimize stranded passengers)
        fitness_scores = [(ind, evaluate_timetable(ind)) for ind in population]
        fitness_scores.sort(key=lambda x: x[1])
        
        if fitness_scores[0][1] < best_fitness:
            best_overall = fitness_scores[0][0]
            best_fitness = fitness_scores[0][1]
            
        # Elitism: Keep top 10%
        next_gen = [ind for ind, _ in fitness_scores[:int(pop_size * 0.1)]]
        
        # Reproduction
        while len(next_gen) < pop_size:
            # Tournament selection
            t1 = random.sample(fitness_scores, 3); t1.sort(key=lambda x: x[1])
            t2 = random.sample(fitness_scores, 3); t2.sort(key=lambda x: x[1])
            
            child1, child2 = crossover(t1[0][0], t2[0][0])
            next_gen.append(mutate(child1))
            if len(next_gen) < pop_size:
                next_gen.append(mutate(child2))
                
        population = next_gen
        
    return best_overall, best_fitness


# --- Phase 2: Simulated Annealing (SA) Sequential Refinement ---

def scramble_mutation(schedule):
    """Swapping or slightly shifting departure seconds to test neighbors."""
    neighbor = schedule.copy()
    if random.random() < 0.5:
        # Swap two departure times
        idx1, idx2 = random.sample(range(FLEET_SIZE), 2)
        neighbor[idx1], neighbor[idx2] = neighbor[idx2], neighbor[idx1]
    else:
        # Shift a departure time by up to +/- 5 minutes (300 seconds)
        idx = random.randint(0, FLEET_SIZE - 1)
        shift = random.randint(-300, 300)
        new_time = neighbor[idx] + shift
        # Bound it within the time window
        neighbor[idx] = max(START_TIME_SEC, min(END_TIME_SEC, new_time))
    return neighbor

def run_simulated_annealing(initial_schedule, initial_temp=1000.0, cooling_rate=0.95, iterations=500):
    """Executes SA local refinement using a Boltzmann probability distribution."""
    current_schedule = initial_schedule
    current_fitness = evaluate_timetable(current_schedule)
    
    best_schedule = current_schedule.copy()
    best_fitness = current_fitness
    
    temp = initial_temp
    
    for i in range(iterations):
        neighbor = scramble_mutation(current_schedule)
        neighbor_fitness = evaluate_timetable(neighbor)
        
        delta = neighbor_fitness - current_fitness
        
        # Acceptance Criteria
        if delta < 0:
            # Better solution, accept it
            current_schedule = neighbor
            current_fitness = neighbor_fitness
            if current_fitness < best_fitness:
                best_schedule = current_schedule.copy()
                best_fitness = current_fitness
        else:
            # Worse solution, accept with Boltzmann probability
            if temp > 0.1:
                probability = math.exp(-delta / temp)
                if random.random() < probability:
                    current_schedule = neighbor
                    current_fitness = neighbor_fitness
                    
        # Temperature cooling
        temp *= cooling_rate
        
    return best_schedule, best_fitness

# ==========================================
# Main Execution Pipeline
# ==========================================
if __name__ == "__main__":
    print("\n--- Phase 1: Genetic Algorithm ---")
    best_ga_schedule, ga_fitness = run_genetic_algorithm(pop_size=30, generations=30)
    print(f"Best GA Schedule (seconds): {sorted(best_ga_schedule)}")
    print(f"Total Stranded Passengers (GA): {ga_fitness}")
    
    print("\n--- Phase 2: Simulated Annealing (Sequential Refinement) ---")
    best_sa_schedule, sa_fitness = run_simulated_annealing(best_ga_schedule, iterations=500)
    
    print("\n--- Final Results ---")
    final_schedule_sorted = sorted(best_sa_schedule)
    print(f"Optimized Timetable (Departure Seconds): {final_schedule_sorted}")
    
    # Convert seconds back to human-readable format for clarity (HH:MM:SS)
    human_readable = [f"{s // 3600}:{(s % 3600) // 60:02d}:{s % 60:02d}" for s in final_schedule_sorted]
    print(f"Optimized Timetable (HH:MM:SS): {human_readable}")
    print(f"Total Stranded Passengers (Final): {sa_fitness}")



--- Phase 1: Genetic Algorithm ---
Best GA Schedule (seconds): [25207, 25393, 25510, 25675, 25752, 25813, 25847, 26035, 26208, 26384]
Total Stranded Passengers (GA): 8050

--- Phase 2: Simulated Annealing (Sequential Refinement) ---

--- Final Results ---
Optimized Timetable (Departure Seconds): [25200, 25393, 25510, 25674, 25675, 25847, 25999, 26111, 26263, 26384]
Optimized Timetable (HH:MM:SS): ['7:00:00', '7:03:13', '7:05:10', '7:07:54', '7:07:55', '7:10:47', '7:13:19', '7:15:11', '7:17:43', '7:19:44']
Total Stranded Passengers (Final): 8033
